# PauliMasks and PauliSums

In this notebook, we'll walk through some of the basic ways to represent Pauli operators in `workbench_algorithms`. 

## [PauliMask](../autoapi/workbench_algorithms/utils/paulimask.html#workbench_algorithms.utils.paulimask.PauliMask)

`PauliMask`s are `workbench_algorithm`'s internal data representation of single-qubit Pauli operators. 

Unlike other ways to store these operators - such as storing them explicitly as matrices in numpy - `PauliMask`s use a bit-wise representation to indicate the type of Pauli operator acting on each qubit due to the unique algebra associated with multiplying Pauli matrices together. Thanks to this bit-wise representation, manipulating `PauliMasks` is especially efficient!

Let's look at how to build some `PauliMask`s so we can get more comfortable with the notation!

### Creating PauliMasks with Integers

The typical way to instantiate a `PauliMask` is by passing two integers: the `x_mask` and the `z_mask`.

As a quick sidenote, both `WorkBench` and `Python` use little-endian notation, so the least significant bit (corresponding to the top-most qubit) is at the rightmost place in the bitstring.

To understand how these masks are used to represent Pauli operators, imagine a system of two qubits. If we try to represent the Pauli-X operator on the first qubit (`"X0"`), we can think of this as simply noting that a Pauli-X operator is being applied to the top-most (rightmost) qubit (bit). Likewise, if we have a Pauli-Z operator acting on the second qubit (`"Z1"`), we can simply note that a Pauli-Z operator is being applied to the bottom-most (leftmost) qubit (bit). We can efficiently represent these two actions as binary numbers "acting" on our register of qubits. 

If we wish to represent this operator (`"X0 Z1"`), then we have to pass the bit masks `01` and `10` as the `x_mask` and `z_mask` respectively. In integer notation, these two numbers are `1` and `2` repsectively, so we can create our `PauliMask` by passing these two integers:

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from psiqworkbench import Qubits, QPU
from workbench_algorithms.utils.paulimask import PauliMask, PauliSum, pauli_sum_to_numpy

import numpy as np

In [ ]:
xz = PauliMask(1, 2)
print(xz.get_pauli_string())

If we want to represent a Pauli-Y operator, we simply note that we can represent represent `Y` as *both* `X` and `Z` acting on that qubit!

Therefore, if we want to represent the operator `X0 Y1`, we can apply our `x_mask` to both qubits and apply our `z_mask` to just the second qubit -- since both terms are "turned on" for the second qubit, the resulting operation will be a `Y`. We can realize this using the `x_mask` `"11"` (which is the integer `3`) and the `z_mask` `"10"` (which is the integer `2`):

In [ ]:
xy = PauliMask(3, 2)
print(xy.get_pauli_string())

### Creating PauliMasks with PauliStrings

We began with creating `PauliMask`s using integers representing bit masks since that is the underlying data representation of the `PauliMask` class and what allows for very efficient manipulation of these objects. However, sometimes it is more user-friendly to create these `PauliMask`s from strings representing our Pauli operators.

We can also do this easily using the `from_pauli_string` method:

In [ ]:
xy = PauliMask.from_pauli_string("X0 Y1")
print(xy.mask)
print(xy.get_pauli_string())

xyiiiy = PauliMask.from_pauli_string("X0 Y1 Y5")
print(xyiiiy.mask)
print(xyiiiy.get_pauli_string())
assert xyiiiy.mask == PauliMask(35, 34).mask

xyiiiy = PauliMask.from_pauli_string("X0 Y1 I2 I3 I4 Y5")
print(xyiiiy.mask)
print(xyiiiy.get_pauli_string())
assert xyiiiy.mask == PauliMask(35, 34).mask

### Manipulating PauliMasks

We can also do some handy algebra using the `PauliMask` object directly such as multiplying. These manipulations are where the efficiency of these bit mask representations shine. Let's check some basic Pauli algebra! 

We should note though that the `PauliMask` object has no coefficient nor sign, so such algebraic rules are neglected here.

In [ ]:
x = PauliMask(1, 0)
z = PauliMask(0, 1)
y = x * z
print(y.get_pauli_string())

In [ ]:
import time
some_big_operator = PauliMask(122398295302235234624, 1232234634544352536)
another_big_operator = PauliMask(854347345634252822, 235197623462345645782)

start = time.time()
mult = (some_big_operator * another_big_operator).get_pauli_string()
time_to_compute = time.time() - start
print(mult)
print("Time to compute: {} sec".format(time_to_compute))

Now that's _wicked_ fast for some really big operators!!!

We can also do some convenient checks regarding commutation:

In [ ]:
i = PauliMask(0, 0)
print(x.commute_check(z))
print(x.commute_check(x))
print(z.commute_check(i))

We can get the commutator of PauliMask, with (returns a `PauliSum`) or without the phase (returns a `PauliMask`):

In [ ]:
xxy = PauliMask(7,4)
zyx = PauliMask(6,3)

print("Without the phase ", xxy.commutator(zyx).get_pauli_string())
print("With the phase ", xxy.commutator(zyx,drop_phase=False))

### Inspecting Elements of a PauliMask

There are also some great helper functions to access particular elements of a `PauliMask` like getting the qubit indices that the operator acts on nontrivially using `get_indices()`. 

Likewise, we can also check a specific qubit index to see what Pauli is acting on the qubit at the given index using `get_pauli()`.

In [ ]:
print(xyiiiy.get_indices())

for index in range(6):
    pauli = xyiiiy.get_pauli(index)
    print("Operator acting on Qubit {}: ".format(index), pauli)

## [PauliSum](../autoapi/workbench_algorithms/utils/paulimask.html#workbench_algorithms.utils.paulimask.PauliSum)

As the name suggests, `PauliSum`s are `workbench_algorithm`s way to represent sums of `PauliMasks`.

### Building PauliSums

Two common ways to construct `PauliSum`s are by adding multiple `PauliMask`s together or by passing them into the constructor along with their coefficients:

In [ ]:
x_plus_y = x + y
print(x_plus_y)
print(pauli_sum_to_numpy(x_plus_y))

x_plus_y = PauliSum(
    [1, x],
    [1, y],
)
print("\n", x_plus_y)
print(pauli_sum_to_numpy(x_plus_y))

As you see above, `workbench_algorithms` also provides a function `pauli_sum_to_numpy()` so that we can easily display the `PauliSum` in matrix format :D

### Getting Some Useful Information About a PauliSum

There are also some helpful methods to get some useful information about a `PauliSum` object.

Some commonly used ones are:
- `wires()`: counts the number of qubits that the `PauliSum` spans across (including qubits only acted on by the identity operator)
- `width()`: counts the number of qubits acted on non-trivially
- `norm()`: calculates the norm (sum of the magnitude of the coefficients) of the `PauliSum`

In [ ]:
pauli_sum = PauliSum(
    [1, PauliMask.from_pauli_string("Z0 X5 Y11")],
    [-15, PauliMask.from_pauli_string("Y0 Z4")],
    [0.25, PauliMask.from_pauli_string("Z5 Z8 Y9 X12")],
    [0.25, PauliMask.from_pauli_string("Z3 Z8")],
    [(1/3), PauliMask.from_pauli_string("X3 Z7 Z8 X9 Y11")],
)
print(pauli_sum)
print("Number of Wires: ", pauli_sum.wires())
print("Width: ", pauli_sum.width())
print("Norm: ", pauli_sum.norm())

We can also get a list of the coefficients using `get_coefficients()` and get the assoicated amplitudes using `get_padded_amplitudes()` - these are the coefficients divided by the norm and also padded so that the length of the list is an integer power of 2 - which can come in handy when building operations such as `PREPARE` and `SELECT`.

In [ ]:
print(pauli_sum.get_coefficients())
print(pauli_sum.get_padded_abs_amplitudes())

### Manipulating PauliSums

There are also lots of helper methods to perform some basic manipulations of `PauliSum`s such as:
- normalizing the operator with `normalize()`
- adding an identity offset with `add_identity_offset()`
- removing all terms with coefficients below some threshold with `remove_below()`
- getting the commutator of two `PauliSum` with `commutator()`

In [ ]:
print("Unnormalized:\n", pauli_sum)
print("Normalized:\n", pauli_sum.normalize())

In [ ]:
coefficient, shifted_sum = pauli_sum.add_identity_offset()
print(shifted_sum)

In [ ]:
print(shifted_sum.remove_below(1))

In [ ]:
print(pauli_sum.commutator(xxy+zyx))